# 03 · 임베딩 · 청킹 · 색인 · 검색 비교
**블록 3 · 고르기 · RAG (오전)** — 실습 30분

RAG의 다섯 단계 — 자르기 → 벡터로 바꾸기 → 저장 → **검색** → 주입해서 묻기.
새로 배우는 것은 검색 하나이고, 나머지는 어제 배운 프롬프트입니다.

> 임베딩 모델은 처음 실행할 때 내려받습니다(약 470MB). 강의장에서 막히면 `.env`에 `EMBED=hash`를 넣고 커널을 다시 시작합니다.

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")
from src import rag
print("임베딩:", "간이 해시 (EMBED=hash)" if rag.EMBED == "hash" else rag.ST_MODEL)
import numpy as np

## 1. 임베딩 — 뜻이 비슷하면 숫자도 가깝다

In [ ]:
docs = ["연차는 사용 3일 전까지 팀장 승인을 받는다",
        "출장비는 귀임 후 7일 이내 정산한다",
        "비밀번호는 90일마다 바꾼다"]
doc_vecs = rag.embed(docs)
print("벡터 모양:", doc_vecs.shape)

for q in ["휴가 신청은 어떻게 해요?", "교통비 돌려받는 법", "암호 변경 주기"]:
    sims = doc_vecs @ rag.embed(q)          # 정규화했으므로 내적 = 코사인 유사도
    best = int(np.argmax(sims))
    print(f"{q:<16} → {docs[best][:20]:<22} " + "  ".join(f"{s:.2f}" for s in sims))

## 2. 청킹 — 어떻게 자를 것인가
검색되는 단위가 곧 모델이 보는 단위입니다. **제목 · 조항 앞에서 먼저 자르고**, 긴 것만 다시 나눕니다.

In [ ]:
import inspect
print(inspect.getsource(rag.chunk).split('"""')[0] + "...")

chunks = rag.chunk_folder("data/kb", max_len=800, overlap=100)
print(f"\n20편 → 조각 {len(chunks)}개 · 가장 긴 조각 {max(len(c['text']) for c in chunks)}자")

### 조각 10개를 무작위로 뽑아 사람이 읽는다
- 혼자 읽어도 뜻이 통하는가
- 어느 문서 어느 절인지 알 수 있는가

In [ ]:
import random
random.seed(7)                     # ✏️ 숫자를 바꾸면 다른 조각이 나온다
for c in random.sample(chunks, 10):
    print(f"── {c['source']} · {c['title'][:30]}")
    print(c["text"][:150].replace("\n", " "), "\n")

## 3. 색인 — 벡터 DB에 넣는다
조각 · 벡터 · 출처를 함께 저장합니다. 출처가 있어야 나중에 각주를 달 수 있습니다.

In [ ]:
idx = rag.Index.build(chunks)
print("저장한 조각:", idx.col.count())

for h in idx.search("연차는 언제까지 신청해요?", k=3):
    print(h)                       # 거리 · 작을수록 가깝다

메타데이터 필터 — "이 문서 안에서만" 같은 조건을 겁니다.

In [ ]:
for h in idx.search("기준 금액", k=3, where={"파일": "10_구매_절차.txt"}):
    print(h)

## 4. 실습 · 키워드 검색과 의미 검색 비교
네 종류의 질문을 두 방식으로 검색하고 1위를 비교합니다. ✏️ 질문을 내 것으로 바꿔 봅니다.

In [ ]:
kw = rag.Keyword(chunks)

TRIALS = [
    ("뜻은 같고 단어가 다른 질문",  "휴가 쓰려면 며칠 전에 말해야 해요?"),
    ("문서에 쓰인 단어 그대로",     "연차 사용 절차"),
    ("코드 · 번호가 들어간 질문",   "AX-2041 정격 토크"),
    ("문서에 답이 없는 질문",       "사내 헬스장 운영 시간"),
]
for kind, q in TRIALS:
    k1, s1 = kw.search(q, 1)[0], idx.search(q, 1)[0]
    print(f"[{kind}] {q}")
    print(f"   키워드 1위  {k1.source:<26} {k1.text[:40].replace(chr(10), ' ')}")
    print(f"   의미   1위  {s1.source:<26} {s1.text[:40].replace(chr(10), ' ')}\n")

| 질문 | 키워드 1위 | 의미 1위 | 맞은 쪽 |
|---|---|---|---|
| 뜻은 같고 단어가 다른 질문 | | | |
| 문서에 쓰인 단어 그대로 | | | |
| 코드 · 번호가 들어간 질문 | | | |
| 문서에 답이 없는 질문 | | | |

**꼭 볼 것** — 마지막 줄. 답이 없는데도 검색은 무언가를 1위로 내놓습니다. **검색은 "없다"고 말하지 않습니다.**
이 성질 때문에 RAG 프롬프트에 거절 지시가 반드시 필요합니다 → 04 노트북.

In [ ]:
# 거리로 "없음"을 가려낼 수 있을까? 답이 있는 질문과 없는 질문의 1위 거리를 나란히 본다
for q in ["연차 사용 절차", "AX-2041 정격 토크", "사내 헬스장 운영 시간", "올해 성과급 지급률"]:
    print(f"{idx.search(q, 1)[0].value:.3f}  {q}")
print("\n→ 경계가 깔끔하게 나뉘나요? 대개는 아닙니다. 그래서 판단은 모델에게 '거절 지시'로 맡깁니다.")